In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.ecommerce_orders_cdf;

**Create the Delta Table and Enable CDF**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.ecommerce_orders_cdf (
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
USING DELTA
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

**Insert Initial Orders**

In [0]:
%sql
INSERT INTO workspace.default.ecommerce_orders_cdf VALUES
(101, 'Arun', 'Pending', 500),
(102, 'Priya', 'Completed', 800),
(103, 'John', 'Pending', 300);

num_affected_rows,num_inserted_rows
3,3


**Verify the data**

In [0]:
%sql
SELECT *
FROM workspace.default.ecommerce_orders_cdf
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Pending,500
102,Priya,Completed,800
103,John,Pending,300


**Update an existing order**

In [0]:
%sql
UPDATE workspace.default.ecommerce_orders_cdf
SET order_status = 'Shipped'
WHERE order_id = 101;

num_affected_rows
1


**Read the changes**

In [0]:
%sql
DESCRIBE HISTORY workspace.default.ecommerce_orders_cdf;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-11T18:17:04.000Z,76591057851675,thiyaguchandran2021@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(991662817728988),37b46f6f-19c4-4b53-aad9-ecf8fcbac7a5,1111-181313-714j6mur-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4173, p25FileSize -> 2401, numDeletionVectorsRemoved -> 1, minFileSize -> 2401, numAddedFiles -> 1, maxFileSize -> 2401, p75FileSize -> 2401, p50FileSize -> 2401, numAddedBytes -> 2401)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T18:17:01.000Z,76591057851675,thiyaguchandran2021@gmail.com,UPDATE,"Map(predicate -> [""(order_id#10399 = 101)""])",null,List(991662817728988),37b46f6f-19c4-4b53-aad9-ecf8fcbac7a5,1111-181313-714j6mur-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 4404, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1625, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2600, rewriteTimeMs -> 2742)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T18:15:54.000Z,76591057851675,thiyaguchandran2021@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(991662817728988),cf4c2a37-18d6-43c9-a5c3-714d06310ed0,1111-181313-714j6mur-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1573)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T18:15:22.000Z,76591057851675,thiyaguchandran2021@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-41266c98-5144-4116-8daa-ae9298c14417"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-a83ded33-62e8-4b70-9eab-e99d789814b1""}, statsOnLoad -> false)",null,List(991662817728988),b5aea62c-19fa-4889-a90a-96ae45aa9a2e,1111-181313-714j6mur-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
ORDER BY _commit_version, _change_type;

order_id,customer_name,order_status,_change_type,_commit_version,_commit_timestamp
103,John,Pending,insert,1,2026-11-11T18:15:54.000Z
101,Arun,Pending,insert,1,2026-11-11T18:15:54.000Z
102,Priya,Completed,insert,1,2026-11-11T18:15:54.000Z
101,Arun,Shipped,update_postimage,2,2026-11-11T18:17:01.000Z
101,Arun,Pending,update_preimage,2,2026-11-11T18:17:01.000Z


**Insert another order and delete an existing order**

Insert order 104

In [0]:
%sql
INSERT INTO workspace.default.ecommerce_orders_cdf VALUES
(104, 'David', 'Pending', 600);

num_affected_rows,num_inserted_rows
1,1


Delete order 103

In [0]:
%sql
DELETE FROM workspace.default.ecommerce_orders_cdf
WHERE order_id = 103;

num_affected_rows
1


Now query the current table.

In [0]:
%sql
SELECT *
FROM workspace.default.ecommerce_orders_cdf
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Shipped,500
102,Priya,Completed,800
104,David,Pending,600


**Read all captured changes**

In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
ORDER BY _commit_version, order_id, _change_type;

order_id,customer_name,order_status,_change_type,_commit_version
101,Arun,Pending,insert,1
102,Priya,Completed,insert,1
103,John,Pending,insert,1
101,Arun,Shipped,update_postimage,2
101,Arun,Pending,update_preimage,2
104,David,Pending,insert,4
103,John,Pending,delete,5


**Version-based incremental processing**

Suppose your pipeline has already processed all changes through Delta version 2

In [0]:
%sql
SELECT *
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    3
);

order_id,customer_name,order_status,amount,_change_type,_commit_version,_commit_timestamp
104,David,Pending,600,insert,4,2026-11-11T18:20:36.000Z
103,John,Pending,300,delete,5,2026-11-11T18:21:20.000Z


**Update Arun's order using MERGE INTO**

In [0]:
%sql
MERGE INTO workspace.default.ecommerce_orders_cdf AS target
USING (
    SELECT
        101 AS order_id,
        'Arun' AS customer_name,
        'Delivered' AS order_status,
        500 AS amount
) AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET
        target.order_status = source.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


**Check the changes captured by CDF**

In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
WHERE order_id = 101
ORDER BY _commit_version, _change_type;

order_id,customer_name,order_status,_change_type,_commit_version,_commit_timestamp
101,Arun,Pending,insert,1,2026-11-11T18:15:54.000Z
101,Arun,Shipped,update_postimage,2,2026-11-11T18:17:01.000Z
101,Arun,Pending,update_preimage,2,2026-11-11T18:17:01.000Z
101,Arun,Delivered,update_postimage,7,2026-11-11T18:26:39.000Z
101,Arun,Shipped,update_preimage,7,2026-11-11T18:26:39.000Z
